<a href="https://colab.research.google.com/github/cbonnin88/mes_allocs-ETL/blob/main/Product_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import polars as pl
import plotly.express as px
from google.cloud import bigquery
from google.colab import auth

# **Authenticate**

In [2]:
auth.authenticate_user()
project_id = 'mes-allocs-analysis'
client = bigquery.Client(project=project_id)

# **Querying the table that I build in dbt**

In [4]:
product_query = """
  SELECT * FROM `mes-allocs-analysis.dbt_mes_allocs.fct_product_events`
"""

df_product = pl.from_pandas(client.query(product_query).to_dataframe())

In [5]:
df_product.head()

user_id,event_type,event_at,device_id,device_type,page_url,session_id,is_conversion,gender,user_status,subscription_plan,plan_price
str,str,"datetime[μs, UTC]",str,str,str,str,bool,str,str,str,f64
"""user_6914""","""application_submitted""",2026-01-01 00:00:08 UTC,"""dev_8929""","""Desktop""","""https://www.mes-allocs.fr/appl…","""174888""",true,"""Female""","""Full-time Employee""","""Free""",0.0
"""user_2843""","""application_submitted""",2026-01-01 00:21:16 UTC,"""dev_3260""","""Desktop""","""https://www.mes-allocs.fr/appl…","""255917""",true,"""Male""","""Job-Seeker""","""Free""",0.0
"""user_11909""","""application_submitted""",2026-01-01 00:38:00 UTC,"""dev_8491""","""Desktop""","""https://www.mes-allocs.fr/appl…","""808932""",true,"""Male""","""Job-Seeker""","""Free""",0.0
"""user_11734""","""application_submitted""",2026-01-01 00:47:21 UTC,"""dev_2603""","""Desktop""","""https://www.mes-allocs.fr/appl…","""611878""",true,"""Non-Binary""","""Job-Seeker""","""Basic""",7.99
"""user_13111""","""application_submitted""",2026-01-01 00:59:43 UTC,"""dev_4694""","""Desktop""","""https://www.mes-allocs.fr/appl…","""479611""",true,"""Female""","""Job-Seeker""","""Free""",0.0


In [6]:
display(df_product.describe())

statistic,user_id,event_type,event_at,device_id,device_type,page_url,session_id,is_conversion,gender,user_status,subscription_plan,plan_price
str,str,str,str,str,str,str,str,f64,str,str,str,f64
"""count""","""600000""","""600000""","""600000""","""600000""","""600000""","""600000""","""600000""",600000.0,"""600000""","""600000""","""600000""",600000.0
"""null_count""","""0""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,"""0""","""0""","""0""",0.0
"""mean""",null,null,"""2026-02-15 01:23:48.920980+00:…",null,null,null,null,0.079595,null,null,null,2.942244
"""std""",null,null,null,null,null,null,null,null,null,null,null,4.64316
"""min""","""user_0""","""application_submitted""","""2026-01-01 00:00:00+00:00""","""dev_1000""","""Desktop""","""https://www.mes-allocs.fr/appl…","""100000""",0.0,"""Female""","""Full-time Employee""","""Basic""",0.0
"""25%""",null,null,"""2026-01-23 13:51:22+00:00""",null,null,null,null,null,null,null,null,0.0
"""50%""",null,null,"""2026-02-15 01:47:57+00:00""",null,null,null,null,null,null,null,null,0.0
"""75%""",null,null,"""2026-03-09 13:08:05+00:00""",null,null,null,null,null,null,null,null,7.99
"""max""","""user_9999""","""subscription_upgraded""","""2026-03-31 23:59:45+00:00""","""dev_9999""","""Tablet""","""https://www.mes-allocs.fr/subs…","""999999""",1.0,"""Non-Binary""","""Part-time Employee""","""Premium""",12.99


# **Metric 1: Retention (Cohort Analysis)**
- I'll group users by their Signup Month and see how many remain active in subsequent months.

In [8]:
# Create Cohorts
cohort_df = df_product.with_columns([
    pl.col('event_at').dt.truncate('1mo').alias('event_month'),
    pl.col('event_at').min().over('user_id').dt.truncate('1mo').alias('cohort_month')
])

display(cohort_df.head())

user_id,event_type,event_at,device_id,device_type,page_url,session_id,is_conversion,gender,user_status,subscription_plan,plan_price,event_month,cohort_month
str,str,"datetime[μs, UTC]",str,str,str,str,bool,str,str,str,f64,"datetime[μs, UTC]","datetime[μs, UTC]"
"""user_6914""","""application_submitted""",2026-01-01 00:00:08 UTC,"""dev_8929""","""Desktop""","""https://www.mes-allocs.fr/appl…","""174888""",true,"""Female""","""Full-time Employee""","""Free""",0.0,2026-01-01 00:00:00 UTC,2026-01-01 00:00:00 UTC
"""user_2843""","""application_submitted""",2026-01-01 00:21:16 UTC,"""dev_3260""","""Desktop""","""https://www.mes-allocs.fr/appl…","""255917""",true,"""Male""","""Job-Seeker""","""Free""",0.0,2026-01-01 00:00:00 UTC,2026-01-01 00:00:00 UTC
"""user_11909""","""application_submitted""",2026-01-01 00:38:00 UTC,"""dev_8491""","""Desktop""","""https://www.mes-allocs.fr/appl…","""808932""",true,"""Male""","""Job-Seeker""","""Free""",0.0,2026-01-01 00:00:00 UTC,2026-01-01 00:00:00 UTC
"""user_11734""","""application_submitted""",2026-01-01 00:47:21 UTC,"""dev_2603""","""Desktop""","""https://www.mes-allocs.fr/appl…","""611878""",true,"""Non-Binary""","""Job-Seeker""","""Basic""",7.99,2026-01-01 00:00:00 UTC,2026-01-01 00:00:00 UTC
"""user_13111""","""application_submitted""",2026-01-01 00:59:43 UTC,"""dev_4694""","""Desktop""","""https://www.mes-allocs.fr/appl…","""479611""",true,"""Female""","""Job-Seeker""","""Free""",0.0,2026-01-01 00:00:00 UTC,2026-01-01 00:00:00 UTC


In [9]:
# Calculate months since signup
cohort_df = cohort_df.with_columns(
    ((pl.col('event_month') - pl.col('cohort_month')).dt.total_days() / 30).round(0).cast(pl.Int64).alias('month_number')
)

display(cohort_df.head())

user_id,event_type,event_at,device_id,device_type,page_url,session_id,is_conversion,gender,user_status,subscription_plan,plan_price,event_month,cohort_month,month_number
str,str,"datetime[μs, UTC]",str,str,str,str,bool,str,str,str,f64,"datetime[μs, UTC]","datetime[μs, UTC]",i64
"""user_6914""","""application_submitted""",2026-01-01 00:00:08 UTC,"""dev_8929""","""Desktop""","""https://www.mes-allocs.fr/appl…","""174888""",true,"""Female""","""Full-time Employee""","""Free""",0.0,2026-01-01 00:00:00 UTC,2026-01-01 00:00:00 UTC,0
"""user_2843""","""application_submitted""",2026-01-01 00:21:16 UTC,"""dev_3260""","""Desktop""","""https://www.mes-allocs.fr/appl…","""255917""",true,"""Male""","""Job-Seeker""","""Free""",0.0,2026-01-01 00:00:00 UTC,2026-01-01 00:00:00 UTC,0
"""user_11909""","""application_submitted""",2026-01-01 00:38:00 UTC,"""dev_8491""","""Desktop""","""https://www.mes-allocs.fr/appl…","""808932""",true,"""Male""","""Job-Seeker""","""Free""",0.0,2026-01-01 00:00:00 UTC,2026-01-01 00:00:00 UTC,0
"""user_11734""","""application_submitted""",2026-01-01 00:47:21 UTC,"""dev_2603""","""Desktop""","""https://www.mes-allocs.fr/appl…","""611878""",true,"""Non-Binary""","""Job-Seeker""","""Basic""",7.99,2026-01-01 00:00:00 UTC,2026-01-01 00:00:00 UTC,0
"""user_13111""","""application_submitted""",2026-01-01 00:59:43 UTC,"""dev_4694""","""Desktop""","""https://www.mes-allocs.fr/appl…","""479611""",true,"""Female""","""Job-Seeker""","""Free""",0.0,2026-01-01 00:00:00 UTC,2026-01-01 00:00:00 UTC,0


In [11]:
# Aggregate
cohort_pivot = cohort_df.group_by(['cohort_month','month_number']).agg(
    pl.col('user_id').n_unique().alias('active_users')
).sort(['cohort_month','month_number'])

display(cohort_pivot)

cohort_month,month_number,active_users
"datetime[μs, UTC]",i64,u32
2026-01-01 00:00:00 UTC,0,15000
2026-01-01 00:00:00 UTC,1,15000
2026-01-01 00:00:00 UTC,2,15000


In [12]:
fig_cohort = px.imshow(
    cohort_pivot.to_pandas().pivot(index='cohort_month',columns='month_number',values='active_users'),
    labels=dict(x='Months Since Signup',y='Cohort Month',color='Active Users'),
    title='Users Retention Cohort',
    color_continuous_scale='Viridis'
)

fig_cohort.show()

# **Metric 2: Conversion Funnel (by Device)**
- Visualizing the drop-off

In [13]:
funnel_data = df_product.group_by(['device_type','event_type']).agg(
    pl.col('user_id').n_unique().alias('user_count')
).filter(pl.col('event_type').is_in(['simulationi_started','simulation_completed','application_submitted']))

display(funnel_data)

device_type,event_type,user_count
str,str,u32
"""Mobile""","""simulation_completed""",8224
"""Mobile""","""application_submitted""",7894
"""Desktop""","""application_submitted""",5736
"""Tablet""","""simulation_completed""",786
"""Desktop""","""simulation_completed""",5950
"""Tablet""","""application_submitted""",760


In [15]:
fig_funnel = px.funnel(
    funnel_data.to_pandas(),
    x='user_count',
    y='event_type',
    color='device_type',
    title='Product Conversion Funnel by Device',
    labels={'user_count':'Number of Users','device_type':'Device Type'}
)

fig_funnel.show()

# **Metrics 3: Feature Adoption (Benefit vs. Status)**
- Which user statuses (Job-Seekers, etc.) are actually selecting benefits?

In [31]:
adoption = df_product.filter(pl.col('event_type') == "benefit_selected") \
                      .group_by(['user_status','subscription_plan']) \
                      .agg(pl.len().alias('selections'))


display(adoption)

user_status,subscription_plan,selections
str,str,u32
"""Job-Seeker""","""Free""",25235
"""Full-time Employee""","""Free""",28244
"""Part-time Employee""","""Free""",9247
"""Full-time Employee""","""Basic""",8284
"""Job-Seeker""","""Premium""",3715
"""Job-Seeker""","""Basic""",6884
"""Part-time Employee""","""Premium""",1243
"""Full-time Employee""","""Premium""",4284
"""Part-time Employee""","""Basic""",2572


In [38]:
fig_feature = px.bar(
    adoption.to_pandas(),
    x='user_status',
    y='selections',
    color='subscription_plan',
    barmode = 'group',
    title='Feature Adoption by User Status',
    color_discrete_sequence=px.colors.sequential.Viridis_r,
    labels= {'selections':'Selection','user_status':'User Status','subscription_plan':'Subscription Plan'}
)

fig_feature.show()

# **Metric 4: Revenue & ARPU**
- Average Revenue Per User by Gender and Status

In [19]:
revenue = df_product.group_by(['user_status','gender']).agg([
    pl.col('plan_price').mean().round(2).alias('arpu')
])

display(revenue)

user_status,gender,arpu
str,str,f64
"""Job-Seeker""","""Male""",2.88
"""Full-time Employee""","""Non-Binary""",2.92
"""Part-time Employee""","""Male""",2.76
"""Part-time Employee""","""Non-Binary""",3.49
"""Job-Seeker""","""Non-Binary""",2.96
"""Job-Seeker""","""Female""",2.93
"""Part-time Employee""","""Female""",2.95
"""Full-time Employee""","""Female""",2.98
"""Full-time Employee""","""Male""",3.01


In [29]:
fig_revenue = px.sunburst(
    revenue.to_pandas(),
    path=['user_status','gender'],
    values = 'arpu',
    color = 'arpu',
    title = 'ARPU Distribution (Status & Gender)',
    color_continuous_scale='Viridis_r'
)

fig_revenue.update_layout(showlegend=False, coloraxis_showscale=False)
fig_revenue.show()